In [ ]:
from train_sinfusion_3d import train_sinfusion

train_sinfusion(
    data_path="train_data.npy",
    outdir="./SinFusion_output",
    epochs=5001,
    batch_size=8,
    timesteps=1000,
    lr=1e-4,
    device="cuda",
)

### **Step 1**. Load Trained Model and Input Data

In [ ]:
import torch
import numpy as np
import time
from tqdm import tqdm
import os
from train_sinfusion_3d import UNet3D, DDPM3D

# =======================================
# Device 설정
# =======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# =======================================
# 설정
# =======================================
epoch = 2240
n_samples = 50


# =======================================
# 데이터 로드
# =======================================
hard_data = np.load("hard_data.npy")
well_ji = [(25, 3), (27, 12)]  # (j, i)

orig_data = np.load("original_data.npy")
nan_mask = np.isnan(orig_data)


# =======================================
# 모델 로드
# =======================================
ckpt_path = f"./SinFusion_output/sinfusion_3d_epoch{epoch}.pt"
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

model = UNet3D(
    in_ch=2,
    base_ch=32,
    time_emb_dim=256
).to(device)

ddpm = DDPM3D(
    model,
    image_size=(192, 40, 32),
    timesteps=1000
).to(device)

checkpoint = torch.load(ckpt_path, map_location=device)
ddpm.load_state_dict(checkpoint["model_state_dict"])
ddpm.eval()

print(f"✓ Checkpoint loaded successfully: epoch {epoch}")

### **Step 2**. Generate 3D Samples with Hard Conditioning

In [ ]:

# =======================================
# 샘플 생성
# =======================================
print(f"Generating {n_samples} samples with shape (200, 48, 36)...")

valid_samples = []
generation_times = []

with torch.no_grad():
    for i in tqdm(range(n_samples), desc="Sampling"):
        start_time = time.time()

        sample = ddpm.sample_hard_with_shape(
            orig=hard_data,
            batch_size=1,
            device=device,
            shape=(200, 48, 36),
            well_ji_list=well_ji,
            t_start_ratio=0.3
        )  # (1, 2, D, H, W)

        generation_times.append(time.time() - start_time)

        sample_np = sample.cpu().numpy().squeeze(0)  # (2, 200, 48, 36)

        if nan_mask.shape == sample_np.shape:
            sample_np[nan_mask] = np.nan

        if np.isnan(sample_np).all() or np.all(sample_np == 0):
            print(f"Skipping invalid sample {i + 1}")
            continue

        valid_samples.append(sample_np)


# =======================================
# 시간 통계
# =======================================
if generation_times:
    avg_time = sum(generation_times) / len(generation_times)
    print(f"Average generation time per sample: {avg_time:.3f} seconds")
